## NHANES Dataset Analysis

This notebook focuses on how to use both the predictive model and the conformance inference model using the National Health and Nutrition Examination Survey (NHANES) dataset. The dataset can be downloaded at https://wwwn.cdc.gov/nchs/nhanes/. 

The dataset has undergone preprocessing steps which includes removing unnecessary variables as well as missing data. More info in the data directory.


In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd

import random
import numpy as np
import torch
from pprint import pprint

from data import normalize_value
from data_real import load_data as load_data_real
from train import train_conformance_inference
from train_bc import cv_loop_bc
from metrics import compute_classification_metrics
from utils import get_device

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    confusion_matrix, roc_auc_score, roc_curve, 
    precision_recall_curve, auc
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    confusion_matrix, roc_auc_score, log_loss
)

device = get_device()

random.seed(0)
torch.manual_seed(0)
np.random.seed(0)

We initialize the main parameters used during the training of the predictive model. The model is training using a `n_fold` cross-validation approach. The number of epochs, batch size, learning rate, weight decay are set to 1000, 64, 0.001, and 0.01, respectively. These values ought to be adjusted in accordance with the specific requirements of the problem, potentially through the application of a grid search technique.

The classification model is trained with an early stop of 4 and no delta. The regression model uses an early stop of 6 with a no delta.


In [2]:
n_folds = 5
n_epochs = 300
batch_size = 16
learning_rate = 0.001
weight_decay = 0.01
dropout = 0.2
hidden_sizes = [8, 4, 2]
patience_classification = 4
min_delta_classification = 0.0
patience_regression = 6
min_delta_regression = 0.0
normalize = False
standardize = True

In [3]:
def evaluate_on_test(model, test_data):
    """Evalúa el modelo en el conjunto de test y retorna métricas detalladas"""
    model.eval()
    with torch.no_grad():
        y_pred_proba = model(
            torch.from_numpy(test_data['x']).to(device)
        ).cpu().detach().numpy()
    
    y_pred_bin = (y_pred_proba >= 0.5).astype(int)
    y_true = np.ravel(test_data['y'])
    
    # Métricas básicas
    accuracy = accuracy_score(y_true, y_pred_bin)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average='weighted'
    )
    cm = confusion_matrix(y_true, y_pred_bin)
    
    # Cross-entropy (log loss)
    ce = log_loss(y_true, y_pred_proba)
    
    return {
        'accuracy': accuracy,
        'auc': auc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ce': ce,
        'cm': cm,
    }

In [ ]:
# Clinical measures: body mass index, waist circunference, diastolic and diastolic pressure, pulse, cholesterol
# selected_combination = ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR', 'LBXGH_39']


# Clinical measures:  body mass index, waist circunference, diastolic and diastolic pressure, pulse, cholesterol
# Only metadata
comb_1 = ['RIDAGEYR', 'RIAGENDR']
# Only clinical measures
comb_2 = ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
# Without any biochemical measures
comb_3 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
# Add colesrterol and triglycerides
comb_4 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR']
# Add Glycosylated Hemoglobin
comb_5 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGH']
# Without A1c
comb_6 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU']
# All
comb_7 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU', 'LBXGH']

combinations = [comb_1, comb_2, comb_3, comb_4, comb_5, comb_6, comb_7]

# Lista para guardar los resultados
results_list = []

for i, comb in enumerate(combinations):
    print(f"[{i+1}/{len(combinations)}] Procesando combinación {i}: {len(comb)} features")
    
    file_real = os.path.join("../data", "nhanes_with_metabolic_syndrome_adults.csv")
    data = load_data_real(file_real, comb, 'metabolic_syndrome', 'WTMEC2YR')

    # We divide data into training and test splits in a 80-20 ratio.
    n = len(data['y'])
    indices = random.sample(np.arange(n).tolist(), n)
    train_split = indices[:int((4*n) / 5)]
    test_split = indices[int((4*n) / 5):]

    train_data = {
        'seqn': data['seqn'][train_split],
        'x': data['x'][train_split],
        'y': data['y'][train_split],
        'w': data['w'][train_split],
        'z': data['z'][train_split]
    }

    test_data = {
        'seqn': data['seqn'][test_split],
        'x': data['x'][test_split],
        'o': data['x'][test_split],
        'y': data['y'][test_split],
        'w': data['w'][test_split],
        'z': data['z'][test_split]
    }

    # Scale training data
    if standardize:
        scaler = StandardScaler()
        # Scale train data
        train_data['x'] = scaler.fit_transform(train_data['x'])
        # Scale test data
        test_data['x'] = scaler.transform(test_data['x'])
        
    k_fold = KFold(n_splits=n_folds, shuffle=False)
    indexes = sorted(range(len(train_data['y']) - 1))
    splits = k_fold.split(indexes)

    model, train_metrics = cv_loop_bc(
        data=train_data,
        splits=splits,
        n_epochs=n_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        patience=patience_classification,
        min_delta=min_delta_classification,
        dropout=dropout,
        hidden_sizes=hidden_sizes
    )
    
    test_metrics = evaluate_on_test(model, test_data)
    
    # Guardar resultados
    result_row = {
        'Combination': i,
        'N_Features': len(comb),
        'Features': ', '.join(comb),
        # Train metrics
        'Train_AUC': float(train_metrics['auc']),
        'Train_Accuracy': float(train_metrics['accuracy']),
        'Train_Precision': float(train_metrics['precision']),
        'Train_Recall': float(train_metrics['recall']),
        'Train_F1': float(train_metrics['f1']),
        'Train_CE': float(train_metrics['ce']),
        # Test metrics
        'Test_AUC': float(test_metrics['auc']),
        'Test_Accuracy': float(test_metrics['accuracy']),
        'Test_Precision': float(test_metrics['precision']),
        'Test_Recall': float(test_metrics['recall']),
        'Test_F1': float(test_metrics['f1']),
        'Test_CE': float(test_metrics['ce'])
    }
    
    results_list.append(result_row)
    
    print(f"  ✓ Train - AUC: {train_metrics['auc']:.4f}, Acc: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}, CE: {train_metrics['ce']:.4f}")
    print(f"  ✓ Test  - AUC: {test_metrics['auc']:.4f}, Acc: {test_metrics['accuracy']:.4f}, F1: {test_metrics['f1']:.4f}, CE: {test_metrics['ce']:.4f}")
    print()

# Crear DataFrame con resultados
results_df = pd.DataFrame(results_list)

# Mostrar tabla resumida
print("\n" + "="*120)
print("RESULTADOS COMPARATIVOS")
print("="*120)
display(results_df[[
    'Combination', 'N_Features',
    'Train_AUC', 'Train_Accuracy', 'Train_F1', 'Train_CE',
    'Test_AUC', 'Test_Accuracy', 'Test_F1', 'Test_CE'
]].style.format({
    'Train_AUC': '{:.4f}',
    'Train_Accuracy': '{:.4f}',
    'Train_F1': '{:.4f}',
    'Train_CE': '{:.4f}',
    'Test_AUC': '{:.4f}',
    'Test_Accuracy': '{:.4f}',
    'Test_F1': '{:.4f}',
    'Test_CE': '{:.4f}'
}).background_gradient(subset=['Test_AUC'], cmap='RdYlGn'))

# Guardar a CSV
results_df.to_csv('comparison_results_nn_2.csv', index=False)
print(f"\n✓ Resultados guardados en 'comparison_results.csv'")


[1/7] Procesando combinación 0: 2 features


/opt/anaconda3/envs/diabi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  ✓ Train - AUC: 0.6998, Acc: 0.6557, F1: 0.0193, CE: 5.5059
  ✓ Test  - AUC: 0.6902, Acc: 0.6682, F1: 0.5353, CE: 0.5928

[2/7] Procesando combinación 1: 7 features
  ✓ Train - AUC: 0.8601, Acc: 0.7787, F1: 0.5832, CE: 2.5454
  ✓ Test  - AUC: 0.8540, Acc: 0.7811, F1: 0.7772, CE: 0.4774

[3/7] Procesando combinación 2: 9 features
  ✓ Train - AUC: 0.8560, Acc: 0.7845, F1: 0.6280, CE: 2.0469
  ✓ Test  - AUC: 0.8367, Acc: 0.7703, F1: 0.7753, CE: 0.4815

[4/7] Procesando combinación 3: 11 features
  ✓ Train - AUC: 0.9230, Acc: 0.8364, F1: 0.7057, CE: 1.8507
  ✓ Test  - AUC: 0.9152, Acc: 0.8245, F1: 0.8151, CE: 0.4092

[5/7] Procesando combinación 4: 12 features
  ✓ Train - AUC: 0.9197, Acc: 0.8415, F1: 0.7047, CE: 1.8271
  ✓ Test  - AUC: 0.9221, Acc: 0.8405, F1: 0.8344, CE: 0.3937

[6/7] Procesando combinación 5: 12 features
  ✓ Train - AUC: 0.9410, Acc: 0.8469, F1: 0.6941, CE: 1.9566
  ✓ Test  - AUC: 0.9385, Acc: 0.8459, F1: 0.8377, CE: 0.3956

[7/7] Procesando combinación 6: 13 features


,Combination,N_Features,Train_AUC,Train_Accuracy,Train_F1,Train_CE,Test_AUC,Test_Accuracy,Test_F1,Test_CE
0,0,2,0.6998,0.6557,0.0193,5.5059,0.6902,0.6682,0.5353,0.5928
1,1,7,0.8601,0.7787,0.5832,2.5454,0.8540,0.7811,0.7772,0.4774
2,2,9,0.8560,0.7845,0.6280,2.0469,0.8367,0.7703,0.7753,0.4815
3,3,11,0.9230,0.8364,0.7057,1.8507,0.9152,0.8245,0.8151,0.4092
4,4,12,0.9197,0.8415,0.7047,1.8271,0.9221,0.8405,0.8344,0.3937
5,5,12,0.9410,0.8469,0.6941,1.9566,0.9385,0.8459,0.8377,0.3956
6,6,13,0.9344,0.8596,0.7561,1.3342,0.9332,0.8614,0.8624,0.3798



✓ Resultados guardados en 'comparison_results.csv'


In this instance, we employ a distinct combination of input variables, with the diabetes variable serving as the class variable. The NHANES dataset is partitioned into training and test sets in a 75/25 ratio. 